<a href="https://colab.research.google.com/github/nohaelkachach/casiav2-splicing-gradcam-audit/blob/main/04_gradcam_iou_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Grad-CAM generation and IoU computation

Generates Grad-CAM heatmaps for all 1,828 spliced images (held out from classifier training — see notebook 03) using both trained classifiers, then computes single-threshold IoU (primary outcome) and AUC-IoU (threshold-free robustness check) against each ground-truth mask.

**Design decisions:**
- Grad-CAM always computed for the fixed "Tampered" class, regardless of the classifier's actual prediction.
- Target layer: `layer4[-1]` (ResNet18), `features[-1]` (EfficientNet-B0).
- IoU binarizes each heatmap at its own mean activation (dataset-adaptive threshold).
- AUC-IoU integrates IoU across thresholds 0.1–0.9 (trapezoidal rule), per Aksoy (2025).

## Setup — mount Drive, load the leak-safe split from notebook 03

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

base = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_revised"  # adjust to your actual path
tp_dir = os.path.join(base, "Tp")
gt_dir = "/content/drive/MyDrive/CASIA2.0/CASIA2.0_Groundtruth/CASIA2.0_Groundtruth"  # adjust if the real path differs

with open("/content/drive/MyDrive/CASIA2.0/casia_split.json") as f:
    split_dict = json.load(f)

spliced_files = split_dict["held_out_spliced"]
print(f"Spliced-only file count: {len(spliced_files)}")  # expect 1828

def get_mask_path(image_filename):
    stem = os.path.splitext(image_filename)[0]
    return os.path.join(gt_dir, f"{stem}_gt.png")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spliced-only file count: 1828
Device: cuda


## Leak check — confirm spliced images were never in classifier train/val
Fails loudly rather than silently proceeding if notebook 03's split was not the leak-safe version.

In [ ]:
train_val_files = set(split_dict["train_tampered"]) | set(split_dict["val_tampered"])
overlap = set(spliced_files) & train_val_files
assert len(overlap) == 0, f"Leak: {len(overlap)} spliced images were in classifier train/val"
print(f"Confirmed: 0 of {len(spliced_files)} spliced images were in classifier train/val.")

Confirmed: 0 of 1828 spliced images were in classifier train/val.


## Copy spliced images + masks to local disk (speeds up the loop below)
Same rationale as notebook 03 — avoids per-file Drive I/O latency across 1,828 × 2 classifiers.

In [ ]:
import shutil, time

local_tp = "/content/Tp_local"
os.makedirs(local_tp, exist_ok=True)
t0 = time.time()
for fname in spliced_files:
    dst = os.path.join(local_tp, fname)
    if not os.path.exists(dst):
        shutil.copy2(os.path.join(tp_dir, fname), dst)
print(f"Copied {len(spliced_files)} spliced images in {time.time()-t0:.1f}s")
tp_dir = local_tp

# Ground-truth masks stay on Drive (gt_dir unchanged) — smaller/less frequent
# reads than the source images, so left as-is unless this also proves slow.

KeyboardInterrupt: 

## Load trained classifiers (checkpoints from notebook 03)

In [ ]:
resnet = models.resnet18(weights=None)
resnet.fc = nn.Linear(resnet.fc.in_features, 2)
resnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_resnet_best.pt", map_location=device))
resnet = resnet.to(device)
resnet.eval()

effnet = models.efficientnet_b0(weights=None)
num_features = effnet.classifier[1].in_features
effnet.classifier[1] = nn.Linear(num_features, 2)
effnet.load_state_dict(torch.load("/content/drive/MyDrive/CASIA2.0/casia_effnet_best.pt", map_location=device))
effnet = effnet.to(device)
effnet.eval()

print("Both checkpoints loaded successfully.")

Both checkpoints loaded successfully.


## Grad-CAM implementation (forward/backward hooks)
Channel-wise gradients are global-average-pooled to give one importance weight per channel; the weighted sum of channels, after ReLU, is the heatmap.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class):
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output[0, target_class]
        score.backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam

gradcam_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## IoU computation — single-threshold (primary) and AUC-IoU (robustness check)

In [ ]:
def _resize_cam(cam, gt_mask_binary):
    return np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize(
        (gt_mask_binary.shape[1], gt_mask_binary.shape[0]), Image.BILINEAR
    )) / 255.0

def compute_iou(cam_resized, gt_mask_binary):
    thresh = cam_resized.mean()
    cam_binary = cam_resized > thresh

    intersection = np.logical_and(cam_binary, gt_mask_binary).sum()
    union = np.logical_or(cam_binary, gt_mask_binary).sum()

    if union == 0:
        return None

    return intersection / union

def compute_auc_iou(cam_resized, gt_mask_binary, thresholds=np.arange(0.1, 1.0, 0.1)):
    ious = []
    for t in thresholds:
        cam_binary = cam_resized > t
        intersection = np.logical_and(cam_binary, gt_mask_binary).sum()
        union = np.logical_or(cam_binary, gt_mask_binary).sum()
        ious.append(intersection / union if union > 0 else 0.0)

    return np.trapz(ious, thresholds)

## Run Grad-CAM + IoU for a given model
Shared routine for both architectures below — target class is always 1 (Tampered).

In [ ]:
def run_gradcam_iou(model, target_layer, model_label):
    gradcam = GradCAM(model, target_layer)
    records = []

    for i, fname in enumerate(spliced_files):
        img_path = os.path.join(tp_dir, fname)
        mask_path = get_mask_path(fname)

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        img_pil = Image.open(img_path).convert("RGB")
        input_tensor = gradcam_transform(img_pil).unsqueeze(0).to(device)

        cam = gradcam.generate(input_tensor, target_class=1)

        orig_size = img_pil.size
        mask_pil = Image.open(mask_path).convert("L").resize(orig_size, Image.NEAREST)
        gt_mask_binary = np.array(mask_pil) > 127

        cam_resized = _resize_cam(cam, gt_mask_binary)
        iou = compute_iou(cam_resized, gt_mask_binary)
        if iou is None:
            continue
        auc_iou = compute_auc_iou(cam_resized, gt_mask_binary)

        records.append({"filename": fname, "iou": iou, "auc_iou": auc_iou})

        if (i + 1) % 200 == 0:
            print(f"[{model_label}] Processed {i+1}/{len(spliced_files)}")

    iou_df = pd.DataFrame(records)
    print(f"\n[{model_label}] Done. Computed IoU for {len(iou_df)} of {len(spliced_files)} spliced images.")
    print(iou_df.describe())
    return iou_df

## Run for ResNet18

In [ ]:
iou_df_resnet = run_gradcam_iou(resnet, resnet.layer4[-1], "ResNet18")
iou_df_resnet.to_csv("/content/drive/MyDrive/CASIA2.0/gradcam_iou_resnet.csv", index=False)

/tmp/ipykernel_3903/2463263135.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(ious, thresholds)


[ResNet18] Processed 200/1828
[ResNet18] Processed 400/1828
[ResNet18] Processed 600/1828
[ResNet18] Processed 800/1828
[ResNet18] Processed 1000/1828
[ResNet18] Processed 1200/1828
[ResNet18] Processed 1400/1828
[ResNet18] Processed 1600/1828
[ResNet18] Processed 1800/1828

[ResNet18] Done. Computed IoU for 1828 of 1828 spliced images.
               iou      auc_iou
count  1828.000000  1828.000000
mean      0.115018     0.081856
std       0.120147     0.088682
min       0.000000     0.000000
25%       0.022594     0.010167
50%       0.078093     0.050593
75%       0.174380     0.128310
max       0.820109     0.479475


## Run for EfficientNet-B0
Target layer is `features[-1]`, not `layer4` (ResNet-specific).

In [ ]:
iou_df_effnet = run_gradcam_iou(effnet, effnet.features[-1], "EfficientNet-B0")
iou_df_effnet.to_csv("/content/drive/MyDrive/CASIA2.0/gradcam_iou_effnet.csv", index=False)

/tmp/ipykernel_3903/2463263135.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(ious, thresholds)


[EfficientNet-B0] Processed 200/1828
[EfficientNet-B0] Processed 400/1828
[EfficientNet-B0] Processed 600/1828
[EfficientNet-B0] Processed 800/1828
[EfficientNet-B0] Processed 1000/1828
[EfficientNet-B0] Processed 1200/1828
[EfficientNet-B0] Processed 1400/1828
[EfficientNet-B0] Processed 1600/1828
[EfficientNet-B0] Processed 1800/1828

[EfficientNet-B0] Done. Computed IoU for 1828 of 1828 spliced images.
               iou      auc_iou
count  1828.000000  1828.000000
mean      0.116666     0.069912
std       0.209910     0.123218
min       0.000000     0.000000
25%       0.000000     0.000339
50%       0.010508     0.007310
75%       0.111468     0.072786
max       0.930004     0.570944


## Merge with extracted features (notebook 02) for the regression
Produces the two regression-ready datasets used in notebook 05.

In [ ]:
features_df = pd.read_csv("/content/drive/MyDrive/CASIA2.0/splice_features.csv")

merged_resnet = features_df.merge(iou_df_resnet, on="filename", how="inner")
merged_resnet.to_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_resnet.csv", index=False)
print(f"ResNet18 regression-ready dataset: {len(merged_resnet)} rows")

merged_effnet = features_df.merge(iou_df_effnet, on="filename", how="inner")
merged_effnet.to_csv("/content/drive/MyDrive/CASIA2.0/regression_ready_effnet.csv", index=False)
print(f"EfficientNet-B0 regression-ready dataset: {len(merged_effnet)} rows")

ResNet18 regression-ready dataset: 1828 rows
EfficientNet-B0 regression-ready dataset: 1828 rows


## Final leak re-check on the merged output
Cheap final safeguard: confirms the actual filenames that made it into the regression-ready CSVs are still disjoint from classifier train/val, in case any merge step silently pulled in extra rows.

In [ ]:
final_overlap_resnet = set(merged_resnet["filename"]) & train_val_files
final_overlap_effnet = set(merged_effnet["filename"]) & train_val_files
assert len(final_overlap_resnet) == 0, f"Leak in final ResNet18 dataset: {len(final_overlap_resnet)} rows"
assert len(final_overlap_effnet) == 0, f"Leak in final EfficientNet-B0 dataset: {len(final_overlap_effnet)} rows"
print("Confirmed: final regression-ready datasets are leak-free.")

Confirmed: final regression-ready datasets are leak-free.
